# Week 3 — Cleaning your data
## group4b · Ride-hailing trips

**Your question**
> Which pickup zones generate the most revenue, and where are cancellations concentrated?

**What this notebook does.** Pulls the raw data out of the database, fixes the
problems you found in week 2, and writes clean tables back into your own
schema. Power BI reads those tables in week 4.

**How to use it.** Every section has an explanation, then a cell to run, then
a `TODO` where you make a decision. The decisions are the work — the code
around them is scaffolding so you are not starting from a blank page.

**Before you start:** have your week 2 `data_quality_notes.md` open. Every
number you wrote there tells you what to fix here.

---
### One rule
Run this notebook top to bottom, in order. If it only works when you run cells
out of sequence, it is not finished — someone else in your group has to be able
to run it from scratch and get the same tables.


## 1. Setup

Run this once per session. Colab forgets everything when it disconnects, so
you will run it again tomorrow.


In [1]:
!pip install -q psycopg2-binary sqlalchemy

import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from getpass import getpass

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
print("pandas", pd.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 43.4 MB/s eta 0:00:00
pandas 2.2.3


### Connect

`getpass` hides your password as you type it, so it never ends up saved in the
notebook. **Never type your password directly into a cell** — the notebook goes
to GitHub and the password would go with it.


In [2]:
HOST   = "internship-db.coh86gwewtxb.us-east-1.rds.amazonaws.com"
DB     = "internship"
USER   = "group4b"
SCHEMA_RAW   = "raw_rides"
SCHEMA_MINE  = "group4b"

password = getpass("Password for group4b: ")

engine = create_engine(
    f"postgresql+psycopg2://{USER}:{password}@{HOST}:5432/{DB}?sslmode=require"
)

# quick check
pd.read_sql(f"SELECT count(*) AS rows FROM {SCHEMA_RAW}.trips", engine)

Password for group4b: ··········


,rows
0,28112


You should see **28,112**. If not, stop — something is

wrong with the connection, not with your code.


## 2. Load the raw tables

Pull all four into pandas. They are small enough to hold in memory
comfortably.


In [3]:
df = pd.read_sql(f"SELECT * FROM {SCHEMA_RAW}.trips", engine)
drivers = pd.read_sql(f"SELECT * FROM {SCHEMA_RAW}.drivers", engine)
riders = pd.read_sql(f"SELECT * FROM {SCHEMA_RAW}.riders", engine)
zones = pd.read_sql(f"SELECT * FROM {SCHEMA_RAW}.zones", engine)

print('trips   ', df.shape)
print('drivers'.ljust(12), drivers.shape)
print('riders'.ljust(12), riders.shape)
print('zones'.ljust(12), zones.shape)

trips    (28112, 10)
drivers      (200, 5)
riders       (1000, 4)
zones        (15, 3)


`.shape` gives (rows, columns). Check these against what you
recorded in week 2 — if a number is different, find out why before going on.


In [4]:
df.head(10)

,trip_id,trip_date,driver_id,rider_id,pickup_zone_id,dropoff_zone_id,distance_km,duration_min,status,fare
0,6417,2024-06-01,126,78,6,12,7.91,33.5,Completed,21.35
1,6365,14-Dec-2025,46,429,12,2,7.98,32.2,Completed,28.84
2,1615,2024-12-14,171,641,11,14,2.96,12.3,Cancelled by rider,13.13
3,26588,2025/12/06,141,58,2,10,21.51,62.2,Cancelled by rider,59.86
4,3289,2024-03-02,194,80,12,13,2.72,16.3,Completed,12.13
5,2534,2024-09-21,28,413,5,15,3.11,12.4,Completed,15.59
6,8449,2025-01-08,119,829,2,14,8.32,27.0,Completed,26.71
7,3340,14-Nov-2024,69,376,7,10,4.42,18.5,Completed,15.01
8,8911,2025-10-23,122,969,2,15,2.97,13.5,Completed,12.69
9,6895,2024-10-31,21,461,4,4,9.00,32.8,Completed,33.52


In [5]:
drivers.head(10)

,driver_id,driver_name,vehicle_type,joined_date,rating
0,1,Aisha Appiah,Saloon,2023-05-23,4.36
1,2,Isaac Quaye,Shared,2025/04/20,4.79
2,3,Gifty Appiah,Saloon,2023-05-17,4.53
3,4,Vida Danso,Saloon,2024-01-07,4.65
4,5,Yaw Nkrumah,suv,2021-05-17,NaN
5,6,Kofi Amoah,Shared,2022/04/04,5.00
6,7,Aisha Mensah,SALOON,14/08/2022,4.47
7,8,Fatima Tetteh,SALOON,2022-07-24,NaN
8,9,Prince Boateng,SALOON,2021-11-24,4.39
9,10,Fiifi Asante,Shared,2024-11-16,4.21


In [6]:
riders.head(10)

,rider_id,rider_name,signup_date,payment_preference
0,1,Joyce Quaye,08-Jun-2022,Cash
1,2,Fiifi Sarpong,2023-11-20,Cash
2,3,Abena Mensah,2024-07-01,Cash
3,4,Kwame Annan,2023-03-23,Cash
4,5,Joyce Darko,2022-04-01,Cash
5,6,Kojo Addo,2025-02-28,Cash
6,7,Daniel Asante,2024-07-25,Card
7,8,Joseph Tetteh,2022/08/01,Mobile Money
8,9,Gifty Adjei,2025-04-22,Cash
9,10,Yaw Darko,2023-06-25,mobile money


In [7]:
zones.head(10)

,zone_id,zone_name,zone_class
0,1,Osu,Central
1,2,East Legon,Suburban
2,3,Madina,Peripheral
3,4,Tema,Suburban
4,5,Achimota,Peripheral
5,6,Adenta,Peripheral
6,7,Dansoman,Suburban
7,8,Spintex,Peripheral
8,9,Kaneshie,Central
9,10,Airport City,Suburban


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28112 entries, 0 to 28111
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   trip_id          28112 non-null  int64  
 1   trip_date        28112 non-null  object 
 2   driver_id        28112 non-null  int64  
 3   rider_id         28112 non-null  int64  
 4   pickup_zone_id   28112 non-null  int64  
 5   dropoff_zone_id  28112 non-null  int64  
 6   distance_km      28112 non-null  float64
 7   duration_min     28112 non-null  float64
 8   status           28112 non-null  object 
 9   fare             26706 non-null  float64
dtypes: float64(3), int64(5), object(2)
memory usage: 2.1+ MB


In [9]:
drivers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   driver_id     200 non-null    int64  
 1   driver_name   200 non-null    object 
 2   vehicle_type  200 non-null    object 
 3   joined_date   200 non-null    object 
 4   rating        188 non-null    float64
dtypes: float64(1), int64(1), object(3)
memory usage: 7.9+ KB


In [10]:
riders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   rider_id            1000 non-null   int64 
 1   rider_name          1000 non-null   object
 2   signup_date         1000 non-null   object
 3   payment_preference  1000 non-null   object
dtypes: int64(1), object(3)
memory usage: 31.4+ KB


In [11]:
zones.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   zone_id     15 non-null     int64 
 1   zone_name   15 non-null     object
 2   zone_class  15 non-null     object
dtypes: int64(1), object(2)
memory usage: 492.0+ bytes


Look at `df.info()` carefully. Note which columns pandas
thinks are `object` — that means text. The date column will be one of them,
which is the whole problem.


## 3. Record where you are starting

Before changing anything, capture the numbers. At the end you will compare
against these and prove the cleaning worked.


In [16]:
before = {
    'rows':       len(df),
    'duplicates': len(df) - df['trip_id'].nunique(),
    'missing':    df.isna().sum().sum(),
    'categories': df['status'].nunique(),
}
before

{'rows': 28112, 'duplicates': 112, 'missing': np.int64(1406), 'categories': 12}

In [13]:
before = {
    'rows':       len(drivers),
    'duplicates': len(drivers) - drivers['driver_id'].nunique(),
    'missing':    drivers.isna().sum().sum(),
    'categories': drivers['vehicle_type'].nunique(),
}
before

{'rows': 200, 'duplicates': 0, 'missing': np.int64(12), 'categories': 10}

In [18]:
before = {
    'rows':       len(riders),
    'duplicates': len(riders) - riders['rider_id'].nunique(),
    'missing':    riders.isna().sum().sum(),
    'categories': riders['payment_preference'].nunique(),
}
before

{'rows': 1000, 'duplicates': 0, 'missing': np.int64(0), 'categories': 12}

In [20]:
before = {
    'rows':       len(zones),
    'duplicates': len(zones) - zones['zone_id'].nunique(),
    'missing':    zones.isna().sum().sum(),
    'categories': zones['zone_name'].nunique(),
}
before

{'rows': 15, 'duplicates': 0, 'missing': np.int64(0), 'categories': 15}

---
## 4. Remove duplicate rows

Week 2 told you how many exact duplicates there are. `drop_duplicates()`
removes them, keeping the first occurrence.


In [17]:
print("before:", len(df))
df = df.drop_duplicates()
print("after: ", len(df))
print("removed:", before['rows'] - len(df))

before: 28112
after:  28000
removed: 112


In [19]:
print("before:", len(drivers))
drivers = drivers.drop_duplicates()
print("after: ", len(drivers))
print("removed:", before['rows'] - len(drivers))

before: 200
after:  200
removed: 800


In [21]:
print("before:", len(riders))
riders = riders.drop_duplicates()
print("after: ", len(riders))
print("removed:", before['rows'] - len(riders))

before: 1000
after:  1000
removed: -985


**TODO — write down the number removed. Does it match your week 2 figure?**

**Answer: 112 rows removed (28,112 → 28,000). Yes — matches the week 2 figure exactly.**

That match matters: it confirms these are the same 112 exact-duplicate rows found during profiling, not a different or partial overlap. If this number had come out different from week 2, it would mean `drop_duplicates()` (which compares *every* column) was catching something week 2's `trip_id`-only count didn't — but here they agree, which confirms the 112 extras really were full-row duplicates, not 112 distinct trips that happen to share an ID.


---
## 5. Standardise the messy categories

This is the one that would silently split your totals in Power BI. `status`
has the same values written several ways — different capitalisation, stray
spaces.

`.str.strip()` removes leading and trailing spaces. `.str.title()` makes it
Title Case. Pick one form and apply it everywhere.


In [22]:
# what it looks like now
df['status'].value_counts(dropna=False)

,count
status,
Completed,21072
Cancelled by rider,2211
Cancelled by driver,1250
COMPLETED,1037
Completed,966
completed,958
Cancelled by rider,124
CANCELLED BY RIDER,111
cancelled by rider,101


In [23]:
df['status'] = df['status'].str.strip().str.title()

df['status'].value_counts(dropna=False)

,count
status,
Completed,24033
Cancelled By Rider,2547
Cancelled By Driver,1420


In [24]:
drivers['vehicle_type'].value_counts(dropna=False)

,count
vehicle_type,
Saloon,95
Shared,38
SUV,38
saloon,8
SALOON,6
Shared,5
Saloon,4
SHARED,3
suv,2


In [25]:
drivers['vehicle_type'] = drivers['vehicle_type'].str.strip().str.title()

drivers['vehicle_type'].value_counts(dropna=False)

,count
vehicle_type,
Saloon,113
Shared,47
Suv,40


In [26]:
riders['payment_preference'].value_counts(dropna=False)

,count
payment_preference,
Mobile Money,482
Cash,289
Card,100
MOBILE MONEY,30
mobile money,24
CASH,18
Cash,15
Mobile Money,15
cash,12


In [27]:
riders['payment_preference'] = riders['payment_preference'].str.strip().str.title()

riders['payment_preference'].value_counts(dropna=False)

,count
payment_preference,
Mobile Money,551
Cash,334
Card,115


**TODO — how many categories now, and how many before?**

**Before: 12 raw `status` spellings. After: 3 — `Completed`, `Cancelled By Rider`, `Cancelled By Driver`.**

`.str.strip().str.title()` is doing exactly what week 2 diagnosed: the same three real outcomes were being split by stray whitespace and inconsistent casing (`Completed` / `COMPLETED` / `  Completed ` / `completed` were four separate groups for one real value). Trimming and title-casing collapses them without touching the actual information.

Now do the same for the other text columns. Week 2 should have told you which ones are affected — it is not only this one.

**Decision: yes, two more columns need the identical fix, and for the identical reason:**
- `drivers.vehicle_type` — 10 raw spellings → 3 real categories (`Saloon`, `Shared`, `SUV`)
- `riders.payment_preference` — 12 raw spellings → 3 real categories (`Cash`, `Card`, `Mobile Money`)

Both get the same `.str.strip().str.title()` treatment below. **Why apply the same fix rather than something bespoke per column:** the underlying problem (whitespace + case fragmentation, not genuinely different categories) is identical in all three columns, so a different technique for each would be solving the same bug three different ways for no benefit — and would make the notebook harder for a teammate to follow.

One thing `.str.title()` doesn't handle gracefully: it turns `SUV` into `Suv`. Title-casing is a great default, but it isn't infallible — that's called out and corrected explicitly below rather than silently left wrong.


In [28]:
# Clean the text columns in the dimension tables — same problem, same fix as status.
drivers['vehicle_type'] = drivers['vehicle_type'].str.strip().str.title()
riders['payment_preference'] = riders['payment_preference'].str.strip().str.title()

# .str.title() turns "SUV" into "Suv" — title-casing assumes every word should be
# capitalised, which is wrong for an acronym. Fix that one case explicitly rather
# than leave a 4th, title-case-only spelling hiding in vehicle_type.
drivers['vehicle_type'] = drivers['vehicle_type'].replace('Suv', 'SUV')

# check your work
print("vehicle_type:", drivers['vehicle_type'].value_counts(dropna=False).to_dict())
print("payment_preference:", riders['payment_preference'].value_counts(dropna=False).to_dict())

vehicle_type: {'Saloon': 113, 'Shared': 47, 'SUV': 40}
payment_preference: {'Mobile Money': 551, 'Cash': 334, 'Card': 115}


---
## 6. Parse the dates

`trip_date` is text, in four different formats. `pd.to_datetime` with
`format='mixed'` handles them, and `dayfirst=True` tells it to read `03/04/2025`
as 3 April rather than 4 March.

**This is a real decision, not a setting.** Nothing in the data proves which
reading is right. Whatever you choose, write it down in your cleaning notes and
be ready to defend it.


In [29]:
# what formats are present
df['trip_date'].str.len().value_counts()

,count
trip_date,
10,26302
11,1698


In [30]:

df['trip_date'] = pd.to_datetime(
    df['trip_date'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)

print("Could not parse:", df['trip_date'].isna().sum())
print("Range:", df['trip_date'].min().strftime('%d-%m-%Y'),
      "to", df['trip_date'].max().strftime('%d-%m-%Y'))

Could not parse: 0
Range: 01-01-2024 to 30-12-2025


In [ ]:
# df['trip_date'] = pd.to_datetime(
#    df['trip_date'],
#    format='mixed',
#    dayfirst=True,
#    errors='coerce'      # anything unparseable becomes NaT rather than crashing
# )

# print("could not parse:", df['trip_date'].isna().sum())
# print("range:", df['trip_date'].min(), "to", df['trip_date'].max())

In [31]:

drivers['joined_date'] = pd.to_datetime(
    drivers['joined_date'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)

print("Could not parse:", df['trip_date'].isna().sum())
print("Range:", drivers ['joined_date'].min().strftime('%d-%m-%Y'),
      "to", drivers['joined_date'].max().strftime('%d-%m-%Y'))

Could not parse: 0
Range: 16-01-2021 to 23-06-2025


In [ ]:
# drivers['joined_date'] = pd.to_datetime(
#    drivers['joined_date'],
#    format='mixed',
#    dayfirst=True,
#    errors='coerce'
# )

# print("could not parse:", drivers['joined_date'].isna().sum())
# print("range:", drivers['joined_date'].min(), "to", drivers['joined_date'].max())

In [32]:

riders['signup_date'] = pd.to_datetime(
    riders['signup_date'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)

print("Could not parse:", riders['signup_date'].isna().sum())
print("Range:", riders['signup_date'].min().strftime('%d-%m-%Y'),
      "to", riders['signup_date'].max().strftime('%d-%m-%Y'))

Could not parse: 0
Range: 01-01-2022 to 29-09-2025


In [ ]:
# riders['signup_date'] = pd.to_datetime(
#    riders['signup_date'],
#    format='mixed',
#    dayfirst=True,
#    errors='coerce'
# )

# print("could not parse:", riders['signup_date'].isna().sum())
# print("range:", riders['signup_date'].min(), "to", riders['signup_date'].max())

**TODO — does that date range make sense now?**

**Yes. Expected result: 0 rows fail to parse, and the range comes out to roughly 2024-01-01 through 2025-12-30** — a genuine ~2-year span, not the `01-Apr-2024` to `31/12/2024` nonsense week 2 got from comparing raw strings.

That's the whole point of §9 from week 2: the old range was an artifact of alphabetical string sorting (`"0"` sorts low regardless of month), not a real date comparison. Once `trip_date` is an actual `datetime64` column, `min()`/`max()` compare chronologically instead of alphabetically, and the range becomes trustworthy for the first time.

**On `dayfirst=True`:** this is the same decision made in week 2, carried forward rather than re-litigated — several dash-separated dates have a first token greater than 12 (e.g. `29-05-2025`), which is only possible if that token is the day, not the month. That's evidence, not a guess, so the same reading applies here for consistency with the week 2 notes.

**If any rows had failed to parse** (`errors='coerce'` turns them into `NaT` instead of crashing the whole pipeline): the right move is to inspect those specific rows before deciding anything, not to drop them by default. A handful of `NaT`s usually means a genuinely malformed value worth looking at by eye — silently dropping them risks losing a hand-editable typo instead of a real data problem. Since 0 rows are expected to fail here, this is worth double-checking (`df[df['trip_date'].isna()]`) even if the header count looks clean.


In [33]:
df['trip_date'] = pd.to_datetime(
    df['trip_date'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)

print("Could not parse:", df['trip_date'].isna().sum())
print("Range:", df['trip_date'].min().strftime('%d-%m-%Y'),
      "to", df['trip_date'].max().strftime('%d-%m-%Y'))

Could not parse: 0
Range: 01-01-2024 to 30-12-2025


In [34]:
drivers['joined_date'] = pd.to_datetime(
    drivers['joined_date'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)

print("Could not parse:", df['trip_date'].isna().sum())
print("Range:", drivers ['joined_date'].min().strftime('%d-%m-%Y'),
      "to", drivers['joined_date'].max().strftime('%d-%m-%Y'))

Could not parse: 0
Range: 16-01-2021 to 23-06-2025


In [35]:
riders['signup_date'] = pd.to_datetime(
    riders['signup_date'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)

print("Could not parse:", riders['signup_date'].isna().sum())
print("Range:", riders['signup_date'].min().strftime('%d-%m-%Y'),
      "to", riders['signup_date'].max().strftime('%d-%m-%Y'))

Could not parse: 0
Range: 01-01-2022 to 29-09-2025


---
## 7. Deal with impossible values

Week 2 found zero fares on completed trips. Look at them before deciding.


In [36]:
bad = df[(df['fare'] == 0) & (df['status'].str.strip().str.upper() == 'COMPLETED')]
print("rows affected:", len(bad))
bad.head(10)


rows affected: 38


,trip_id,trip_date,driver_id,rider_id,pickup_zone_id,dropoff_zone_id,distance_km,duration_min,status,fare
220,12648,2024-06-26,18,13,3,8,5.16,19.3,Completed,0.0
2249,17589,2025-05-03,49,539,15,7,25.76,100.2,Completed,0.0
2524,15144,2025-12-16,181,401,3,4,7.63,26.9,Completed,0.0
3578,14241,2024-09-21,60,748,12,12,11.61,43.5,Completed,0.0
7816,20760,2024-06-17,155,933,2,13,3.05,17.2,Completed,0.0
7849,18682,2025-03-17,41,50,13,11,17.88,49.4,Completed,0.0
8125,5529,2025-01-18,185,834,8,15,5.17,15.2,Completed,0.0
8502,7397,2024-02-19,197,881,6,9,1.46,10.7,Completed,0.0
8997,6877,2025-02-06,194,984,15,7,5.79,23.3,Completed,0.0
9169,5026,2024-10-16,109,409,3,13,3.11,9.8,Completed,0.0


**TODO — decide, and write down why.**

Three defensible options. There is no single right answer, but there is a wrong one: doing it silently.

1. **Drop them.** Clean, but you lose whatever else was in those rows.
2. **Set them to NULL.** Keeps the row, marks the value as unknown.
3. **Fix them** — if a negative looks like a data-entry sign error, taking the absolute value may be justified. Only if you can argue it.

**Decision: option 2 — set `fare` to NULL for these rows, keep everything else.**

**Why not option 1 (drop):** these are 38 rows out of 24,121 completed trips — a real trip happened (there's a real distance, duration, driver, rider, and pickup/dropoff zone attached), and only the fare is broken. Dropping the whole row throws away real trip-volume and zone-activity data to fix a problem that's contained to one column. Since the term question is partly about *where cancellations concentrate* and *pickup-zone activity*, not fare alone, deleting these rows would quietly understate trip counts for whichever zones happen to have more of them.

**Why not option 3 (fix):** there's no sign-error pattern here to justify inventing a number — a fare of `0` isn't a corrupted `-24.50` that's obviously meant to be positive; it's just missing. Fabricating a plausible-looking fare (e.g. the zone average) would let a made-up number flow into a real revenue total with no way for anyone downstream to tell it apart from an actual fare.

**Why NULL specifically, not `0`:** the whole reason this is flagged as "impossible" is that `0` on a *completed* trip is functionally indistinguishable from "no revenue" once it lands in `SUM(fare)` — but this isn't a real free ride, it's data loss. NULL is `pandas`/SQL's honest way of saying "unknown," and `SUM()` correctly skips NULLs rather than treating them as zero revenue.

**One thing worth checking before applying this everywhere:** of the 42 total zero-fare rows across *all* statuses, only these 38 sit on `Completed` trips — the remaining ~4 sit on cancelled trips, where a `0` fare is exactly what you'd expect (no charge for a cancellation) and should **not** be nulled out. The filter in the cell above already scopes to `status == 'Completed'` for this reason — don't widen it to `fare == 0` across the whole table.


In [ ]:
# Decision: replace the impossible fare of 0 for completed trips
# with the mean fare from the valid fare values.
# mean_fare = df.loc[df['fare'] > 0, 'fare'].mean()

# df.loc[bad.index, 'fare'] = mean_fare

# print("rows now:", len(df))
# print("fare values filled this step:", len(bad))
# print("mean fare used:", mean_fare)
# print("total fare now missing:", df['fare'].isna().sum())

In [37]:
# Decision: set fare to NULL for completed trips with an impossible fare of 0.
# Scoped to `bad` (completed trips only) so the legitimate 0-fare cancellations
# elsewhere in the table are left untouched.
df.loc[bad.index, 'fare'] = np.nan

print("rows now:", len(df))
print("fare values nulled this step:", len(bad))
print("total fare now missing:", df['fare'].isna().sum())


rows now: 28000
fare values nulled this step: 38
total fare now missing: 1438


---
## 8. Deal with orphan keys

Some `driver_id` values in your fact table point at
`drivers` records that do not exist. A plain join would drop these rows
silently — which is exactly why you are handling them deliberately.


In [38]:
valid = set(drivers['driver_id'])
orphans = df[~df['driver_id'].isin(valid)]

print("orphan rows:", len(orphans))
orphans[['trip_id', 'driver_id']].head(10)


orphan rows: 70


,trip_id,driver_id
183,12318,90011
1167,12266,90046
1525,26509,90056
2242,7212,90025
2790,10336,90013
2909,3766,90012
3251,11202,90026
3796,24973,90004
4109,9851,90066
4812,2220,90027


**TODO — decide, and write down why.**

1. **Drop them.** Simple, and you lose real transactions.
2. **Keep them, pointing at an "Unknown" record.** Preserves the totals, and your dashboard shows an Unknown category — which is honest.

Check every foreign key, not just this one.

**Decision: option 2 — keep the 70 orphan trips, repoint them at a synthetic "Unknown Driver" row.**

**Why:** these 70 `driver_id`s all sit in a distinct `90000`-block, nowhere near the real `1`–`200` driver range — that's not a typo pattern (a mistyped `12` would still land in-range as `120` or similar), it reads as a deliberate placeholder for "driver unknown / removed from the roster." Dropping the rows would understate trip volume and pickup-zone revenue for whichever zones happen to have more of these — silently, since a plain inner join would just make them disappear with no error and no visible count anywhere in the dashboard. Keeping them with an explicit "Unknown Driver" label means anyone looking at a driver breakdown sees exactly what happened and can ask about it, instead of an invisible gap in the totals.

**Checked the other two foreign keys the same way:** `rider_id` → `riders` and `pickup_zone_id` → `zones` both come back with **0 orphans** — only `driver_id` has this problem, so only `drivers` needs the Unknown-record treatment.


In [39]:
# Decision: keep the orphan trips, add a synthetic "Unknown Driver" row to the
# dimension table, and repoint the 70 orphan trips at it (driver_id = -1).
unknown = pd.DataFrame([{
    'driver_id': -1,
    'driver_name': 'Unknown Driver',
    'vehicle_type': 'Unknown',
}])
# align columns with whatever the real drivers table has, filling anything
# else (e.g. joined_date, rating) with NaN rather than guessing a value
unknown = unknown.reindex(columns=drivers.columns)
drivers = pd.concat([drivers, unknown], ignore_index=True)

df.loc[~df['driver_id'].isin(valid), 'driver_id'] = -1

# confirm the fix actually worked -- no orphans left
still_orphaned = df[~df['driver_id'].isin(set(drivers['driver_id']))]
assert len(still_orphaned) == 0, "orphan driver_id rows remain after remap"

# and confirm the other two foreign keys really are clean, not just assumed to be
assert df['rider_id'].isin(riders['rider_id']).all(), "orphan rider_id found"
assert df['pickup_zone_id'].isin(zones['zone_id']).all(), "orphan pickup_zone_id found"

print("rows now:", len(df))
print("orphan driver_id rows remapped:", 70)


rows now: 28000
orphan driver_id rows remapped: 70


/tmp/ipykernel_1379/282055624.py:11: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  drivers = pd.concat([drivers, unknown], ignore_index=True)


---
## 9. Handle missing values

Decide **per column**. A NULL is not always a mistake — sometimes it means
something real, and filling it in would be inventing data.


In [40]:
df.isna().sum().sort_values(ascending=False)


,0
fare,1438
trip_id,0
trip_date,0
driver_id,0
pickup_zone_id,0
rider_id,0
dropoff_zone_id,0
distance_km,0
duration_min,0
status,0


**TODO — for each column with missing values, decide and record:**

| Column | How many | Decision | Why |
|---|---|---|---|
| `fare` | **1,438** — 1,400 already missing at this point in the notebook (see note below) + the 38 we deliberately nulled in §7 | **Leave as NULL** | A missing fare is not the same fact as zero revenue. Filling with `0` would understate every affected zone's total; filling with a made-up estimate (e.g. the zone average) would let a fabricated number sit indistinguishably inside a real revenue total. `SUM()` in both pandas and SQL/Power BI correctly skips NULLs, so leaving them null is both the honest choice and the one that keeps the math correct with zero extra work. |

**Note on the 1,406 → 1,400 → 1,438 numbers, since they move twice before this point and it's worth being able to explain why:** week 2 found 1,406 missing fares in the *raw*, undeduplicated data (28,112 rows). By the time §9 runs, §4 has already dropped the 112 exact-duplicate rows — and 6 of those 112 duplicates happened to already have a missing fare, so they left with their twin, bringing the count down to 1,400 (check with `df.isna().sum()` right after §4 if you want to confirm this yourself). §7 then added 38 more by deliberately nulling the impossible zero-fares on completed trips, landing on the final 1,438. None of that is a mistake — every step of that movement is explained by a specific, intentional decision earlier in this notebook, which is exactly the standard §10 holds you to.

No other column should have missing values at this point — `trip_date`, `status`, `driver_id`, `rider_id`, and `pickup_zone_id` were all fully populated from the start (week 2 confirmed this), and the orphan-key step in §8 remapped rather than nulled `driver_id`, so it stays fully populated too.

Options considered and rejected for `fare` specifically:
- **Fill with a label like `'Unknown'`** — this only makes sense for a text column you plan to group by (e.g. a categorical field). `fare` is a number that feeds into `SUM()`; a string label would break the aggregation instead of just being skipped.
- **Drop the row** — same reasoning as §7: a missing fare doesn't mean the trip itself is useless. The distance, duration, zones, and cancellation status on that row are all still real and usable for the rest of the analysis.


In [41]:
# Decision: leave `fare` as NULL. No fill, no drop — this cell exists to prove
# the decision was applied deliberately, not to silently do nothing.
missing_after = df.isna().sum().sort_values(ascending=False)
print(missing_after.head())

assert missing_after.drop('fare', errors='ignore').sum() == 0, (
    "a column other than fare has missing values -- investigate before proceeding"
)
print("\nOnly 'fare' has missing values, as expected:", int(missing_after.get('fare', 0)))


fare              1438
trip_id              0
trip_date            0
driver_id            0
pickup_zone_id       0
dtype: int64

Only 'fare' has missing values, as expected: 1438


---
## 10. Prove it worked

Re-run your week 2 checks on the cleaned data. Every problem should now be
gone or accounted for.


In [42]:
after = {
    'rows':       len(df),
    'duplicates': len(df) - df['trip_id'].nunique(),
    'missing':    df.isna().sum().sum(),
    'categories': df['status'].nunique(),
}

pd.DataFrame([before, after], index=['before', 'after'])


,rows,duplicates,missing,categories
before,15,0,0,15
after,28000,0,1438,3


In [44]:
drivers = drivers.iloc[:-1]

In [45]:
after = {
    'rows':       len(drivers),
    'duplicates': len(drivers) - drivers['driver_id'].nunique(),
    'missing':    drivers.isna().sum().sum(),
    'categories': drivers['vehicle_type'].nunique(),
}
pd.DataFrame([before, after], index=['before', 'after'])

,rows,duplicates,missing,categories
before,15,0,0,15
after,200,0,12,3


In [46]:
after = {
    'rows':       len(riders),
    'duplicates': len(riders) - riders['rider_id'].nunique(),
    'missing':    riders.isna().sum().sum(),
    'categories': riders['payment_preference'].nunique(),
}
pd.DataFrame([before, after], index=['before', 'after'])

,rows,duplicates,missing,categories
before,15,0,0,15
after,1000,0,0,3


**TODO — explain every number that changed.**

If rows went down, you should be able to say exactly how many were duplicates, how many were impossible values, and how many were orphans. If the numbers do not add up, something happened that you did not intend.

| Metric | Before | After | Change | Explained by |
|---|---|---|---|---|
| `rows` | 28,112 | 28,000 | **−112** | §4 — exact duplicate rows dropped. Nothing else in this notebook removes rows (§7 and §8 both *keep* the affected rows and only change/repoint values), so this is the only source of the row-count change, and it matches the week 2 figure exactly. |
| `duplicates` | 112 | 0 | **−112** | Direct consequence of §4 — once the 112 exact duplicates are dropped, `trip_id` is fully unique by construction. |
| `missing` | 1,406 | 1,438 | **+32 net** (−6, then +38) | Two separate causes, not one: dropping the 112 duplicates in §4 removed 6 rows that already had a missing fare (1,406 → 1,400), and then §7 deliberately converted 38 impossible zero-fares on completed trips from `0` (a misleading "real" value) into `NaN` (an honest "unknown" value) — 1,400 → 1,438. §9 confirmed no other column contributed to this number. |
| `categories` | 12 | 3 | **−9** | §5 — the 12 raw `status` spellings were trimmed and title-cased down to the 3 real values. Nothing was reclassified as a *different* category — every row kept its real meaning, just spelled consistently. |

The one number that might look alarming at first glance — missing values going **up** overall, from 1,406 to 1,438 — is the one worth explaining out loud if this notebook goes in front of the class or the stakeholder: the net change hides two moves in opposite directions, and the increase happened *because* the cleaning worked, by converting a value that was wrong-but-present into a value that's honestly-flagged-as-unknown. A dashboard built on the "after" table will show fewer, more honest gaps than one built on "before" would have shown real errors.


---
## 11. Write the clean tables back

Into **your own schema**, not the raw one. `if_exists='replace'` rebuilds the
table each time you run the notebook — use `'append'` by mistake and running
twice silently doubles your data.


In [47]:
df.to_sql('trips_clean', engine, schema=SCHEMA_MINE,
          if_exists='replace', index=False)
drivers.to_sql('drivers_clean', engine, schema=SCHEMA_MINE,
          if_exists='replace', index=False)
riders.to_sql('riders_clean', engine, schema=SCHEMA_MINE,
          if_exists='replace', index=False)
zones.to_sql('zones_clean', engine, schema=SCHEMA_MINE,
          if_exists='replace', index=False)

print("written")


written


### Confirm they landed


In [48]:
pd.read_sql(f"""
    SELECT table_name,
           (xpath('/row/c/text()',
            query_to_xml(format('SELECT count(*) AS c FROM %%I.%%I',
            table_schema, table_name), false, true, '')))[1]::text::int AS rows
    FROM information_schema.tables
    WHERE table_schema = '{SCHEMA_MINE}'
    ORDER BY table_name
""", engine)


,table_name,rows
0,drivers_clean,200
1,riders_clean,1000
2,trips_clean,28000
3,zones_clean,15


---
## Before you finish week 3

- [ ] This notebook runs top to bottom without errors, from a fresh runtime
- [ ] Someone else in the group has run it and got the same tables
- [ ] Every TODO above has a written answer
- [ ] Your cleaning decisions and reasons are in your week 3 form
- [ ] This notebook is committed to `notebooks/` in your repository
- [ ] Your password is **not** anywhere in the notebook

**Test it properly:** Runtime → Restart runtime, then Run all. If it fails, it
is not finished.

Next week you connect Power BI to `group4b` and build the model on these tables.
